In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch


llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation",
    pipeline_kwargs ={
        "temperature" : 0.5,
        "max_new_tokens" : 100
    }
)

model = ChatHuggingFace(llm = llm)
parser = StrOutputParser()

technical_prompt = PromptTemplate.from_template(
"Give a technical explanation of: {question}\nAnswer:"
)

general_prompt = PromptTemplate.from_template(
"Give a simple general explanation of: {question}\nAnswer:"
)

technical_chain = technical_prompt | model | parser
general_chain = general_prompt | model | parser

router = RunnableBranch(
(
lambda x: "python" in x["question"].lower()
or "java" in x["question"].lower(),
technical_chain,
),
general_chain,
)

result = router.invoke({
"question": "What is Python list comprehension?"
})

print(result)